In [1]:
# conda env 'misr'

import MisrToolkit as mtk
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

In [2]:
def nearest_sample(lat, lon, path, resolution):
    region = mtk.MtkRegion(lat,lon, 1, 1, "km")
    lats, lons = region.snap_to_grid(path, resolution).create_latlon()
    lats = lats.ravel()
    lons = lons.ravel()
    lat_diff = lats - lat
    lon_diff = lons - lon
    D = (lat_diff**2 + lon_diff**2)**0.5
    return lats[np.argmin(D)], lons[np.argmin(D)]

In [3]:
## INPUT
path = 117
orbit = 105254
database = '../../data/MISR'
cameras = ['DF', 'CF', 'BF', 'AF', 'AN', 'AA', 'BA', 'CA', 'DA']
lat, lon = nearest_sample(19.8, 120.4, path, 17600)

## OUTPUT
out_file = '../intermediate_data/MISR_OBS_CAMP2Ex_RF17_Poll1.nc'

In [4]:
grp_general = '{}/MISR_AM1_GRP_ELLIPSOID_GM_P{:d}_O{:d}_{}_F03_0024.hdf'
gmp_general = '{}/MISR_AM1_GP_GMP_P{}_O{}_F03_0013.hdf'
region = mtk.MtkRegion(lat, lon, 8.8*3, 8.8*3, "km")
# region = mtk.MtkRegion(path, 74, 76) # RF17
lat275, lon275 = region.snap_to_grid(path, 275).create_latlon()
lat17600, lon17600 = region.snap_to_grid(path, 17600).create_latlon()
assert lat17600.shape[0] % 2 == 1
assert lon17600.shape[0] % 2 == 1
i_center_lat = int((lat17600.shape[0] - 1) / 2)
j_center_lon = int((lon17600.shape[1] - 1) / 2)

In [5]:
"""
At each MISR sample coordinate, angles are reported in Local North (LN) 
coordinates. That is, +x points North, +y points East, and +z points into the 
Earth and is normal to the ellipsoid. Azimuthal angles are measured in the 
usuial way - from +x in the direction of +y first. Solar zenith angle is 
measured from the +z axis while viewing zenith angles are measured from the -z 
axis. Solar and viewing angles are reported in the direction of photon travel. 
Solar - away from the sun. View - into the sensor.

References:
https://asdc.larc.nasa.gov/documents/misr/DPS_v33_RevM.pdf
https://eospso.gsfc.nasa.gov/sites/default/files/atbd/atbd-misr-03.pdf
"""
gmp_file = gmp_general.format(database, path, orbit)
gmp_grid = mtk.MtkFile(gmp_file).grid('GeometricParameters')
SZA = gmp_grid.field('SolarZenith').read(region).data()
SAA = gmp_grid.field('SolarAzimuth').read(region).data()
VZA = np.zeros(shape=(len(cameras), lat17600.shape[0], lat17600.shape[1]))
VAA = np.zeros(shape=(len(cameras), lat17600.shape[0], lat17600.shape[1]))
for i, c in enumerate(cameras):
    VZA[i,:,:] = gmp_grid.field('{}Zenith'.format(c.lower().capitalize())).read(region).data()
    VAA[i,:,:] = gmp_grid.field('{}Azimuth'.format(c.lower().capitalize())).read(region).data()
SZA_center = SZA[i_center_lat, j_center_lon]
SAA_center = SAA[i_center_lat, j_center_lon]
VZA_center = VZA[:, i_center_lat, j_center_lon]
VAA_center = VAA[:, i_center_lat, j_center_lon]

In [6]:
grp_file = [grp_general.format(database, path, orbit, c.upper()) for c in cameras]
brf = np.zeros(shape=(len(cameras), lat275.shape[0], lat275.shape[1]))
for i, f in enumerate(grp_file):
    brf[i,:,:] = mtk.MtkFile(f).grid('RedBand').field('Red Brf').read(region).data()

In [7]:
misr_obs = xr.Dataset(
                data_vars = {
                    'brf': (['camera', 'y275', 'x275'], brf),
                    'lat275': (['y275', 'x275'], lat275),
                    'lon275': (['y275', 'x275'], lon275),
                    'lat17600': (['y17600', 'x17600'], lat17600),
                    'lon17600': (['y17600', 'x17600'], lon17600),
                    'SZA': (['y17600', 'x17600'], SZA),
                    'SAA': (['y17600', 'x17600'], SAA),
                    'VZA': (['camera', 'y17600', 'x17600'], VZA),
                    'VAA': (['camera', 'y17600', 'x17600'], VAA),
                    'VZA_center': (['camera'], VZA_center),
                    'VAA_center': (['camera'], VAA_center),
                    'SZA_center': SZA_center,
                    'SAA_center': SAA_center,
                    'center_lat': region.center[0],
                    'center_lon': region.center[1],
                },
                coords = {
                    'camera': cameras
                },
                attrs = {
                    'orbit': orbit,
                    'path': path,
                    'blocks': region.block_range(path),
                }
)
misr_obs.to_netcdf(out_file)

In [8]:
misr_obs

<xarray.Dataset>
Dimensions:     (camera: 9, x17600: 3, x275: 192, y17600: 3, y275: 192)
Coordinates:
  * camera      (camera) <U2 'DF' 'CF' 'BF' 'AF' 'AN' 'AA' 'BA' 'CA' 'DA'
Dimensions without coordinates: x17600, x275, y17600, y275
Data variables: (12/15)
    brf         (camera, y275, x275) float64 0.1154 0.1177 ... 0.07053 0.07171
    lat275      (y275, x275) float64 20.1 20.1 20.1 20.1 ... 19.59 19.59 19.59
    lon275      (y275, x275) float64 120.2 120.2 120.2 ... 120.7 120.7 120.7
    lat17600    (y17600, x17600) float64 20.02 20.0 19.99 ... 19.7 19.69 19.67
    lon17600    (y17600, x17600) float64 120.3 120.5 120.6 ... 120.3 120.4 120.6
    SZA         (y17600, x17600) float64 28.92 28.82 28.71 ... 28.68 28.57 28.46
    ...          ...
    VZA_center  (camera) float64 70.21 60.06 45.67 26.16 ... 45.63 60.1 70.42
    VAA_center  (camera) float64 15.67 16.21 17.04 18.98 ... 194.0 194.7 195.2
    SZA_center  float64 28.69
    SAA_center  float64 322.3
    center_lat  float64 19.85
    center_lon  float64 120.5
Attributes:
    orbit:    105254
    path:     117
    blocks:   (75, 75)